In [ ]:
import numpy as np
import cvxpy as cp
import pickle
import gzip

from qiskit.quantum_info import SparsePauliOp

from matplotlib import pyplot as plt

from qphaset.phases import gstates_to_rdms_matrix, phases_vfield
from qphaset.plotting import plot_grad_g_angle_stream

: 

## Order parameter discovery.

In [ ]:
# TODO Consider drho/dlambda (derivative) as other possibility for meas.
# For the derivatives consider for example the case analogous to Magnetization -> Magnetization susceptibility.
# TODO Show effect of fixed angle shift on shift by identity.
# TODO Use linear algebra to solve the PSD problem. Also consider L1 reg with proximal operators.
# TODO Recover vector field using 2 obs (sin, cos).
# TODO Mean field theory mode.
# TODO Extend to Hermitian (currently it is symmetric).

In [ ]:
model_name = "Cluster"
l = 15
n = 10
params = np.linspace(0.5, 0.6, n), np.linspace(0.6, 0.5, n) # upside-down
params = map(lambda m: m.flatten(), np.meshgrid(*params, indexing='xy'))
params = tuple(params)
params = np.stack(params).T
params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

device = 'pc'
device = 'ngt'

if device == 'pc':
    device_path = "D:/code"
elif device == 'ngt':
    device_path = "/eos/user/f/fdimarca"

# dmrg params
chi = 100 # bond dimension
c1 = 1e-3 # eps symm. break.
if model_name == 'ANNNI':
    path_to_tensor = f"{device_path}/projects/2_ANNNI/results/data"
    path_to_figures = f"{device_path}/projects/2_ANNNI/figures"
    axis_name = ('k', 'h')

elif model_name == 'Cluster':
    path_to_tensor = f"{device_path}/projects/3_CLUSTER/results/data"
    path_to_figures = f"{device_path}/projects/3_CLUSTER/figures"
    axis_name = ('k', 'h')

elif model_name == 'Rydberg':
    path_to_tensor = f"{device_path}/projects/4_RYDBERG/results/data"
    path_to_figures = f"{device_path}/projects/4_RYDBERG/figures"
    axis_name = ('$\\Delta/\\Omega$', '$R_b/a$')

else:
    raise SyntaxError("Choose a valid model among 'ANNNI', 'Cluster', and 'Rydberg'")


filename = f'{path_to_tensor}/{model_name}_L_{l}_lambda_1_{params_extent[2]}-{params_extent[3]}_lambda_2_{params_extent[0]}-{params_extent[1]}_npoints_{n}x{n}_chi_{chi}_eps_{c1}.pkl'


with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

In [ ]:
sites = [l // 2]
#sites = [2]
# sites = [l // 2, l // 2 + 1]

rdms = gstates_to_rdms_matrix(gstates, sites=sites)
rdms = rdms[::-1]   # TODO fix

In [ ]:
theta = 0  # Adjust st phases have opposite signs.
# Most of the times theta=0 is good, however, use theta=pi to obtain the complementary
# order parameter. 

In [ ]:
grad_g = phases_vfield(rdms, scale=1)
ys = np.sin(np.angle(grad_g) + theta)

# Labels plot
plt.matshow(np.sign(ys), origin='lower', extent=params_extent)

In [ ]:
rdms = rdms[1:-1, 1:-1] # TODO fix

In [ ]:
lattice_shape = rdms.shape[:2]
rdms = np.reshape(rdms, (-1, ) + rdms.shape[2:])
ys = ys.flatten()

In [ ]:
assert rdms.ndim == 3

# Observable is the optimization variable. For now we consider a symmetric
# operator (real).
obs = cp.Variable(shape=rdms.shape[1:], symmetric=True)

trace_zero_obs = False

def symm_norm(m):
    n = m.shape[0]
    s = [cp.quad_form(m[i, i:], np.eye(n - i)) for i in range(n)]
    return cp.sum(s)

# TODO Review gamma which is now a tradeoff between the two objectives.
# Also see Boyd p. 229 for strong duality of quadratic problems.
gamma = 10

# Construction of the objective.
# TODO Normalize terms by cardinality of corresponding sets.

# Normalization factors.
fp, fn = 1 / np.count_nonzero(ys > 0), 1 / np.count_nonzero(ys < 0)

pos_y_term = -cp.sum([cp.trace(obs @ rdm) for rdm in rdms[np.nonzero(ys > 0)]]) * fp
neg_y_term = cp.sum([cp.square(cp.trace(obs @ rdm)) for rdm in rdms[np.nonzero(ys < 0)]]) * fn
obj = cp.Minimize(pos_y_term + gamma * neg_y_term)


constraints=[symm_norm(obs) <= 1]
if trace_zero_obs:
    constraints.append(cp.trace(obs) == 0)

# Solver
prob = cp.Problem(obj, constraints=constraints)
prob.solve(verbose=True)
obs = obs.value

In [ ]:
plt.matshow(obs)

In [ ]:
np.linalg.eigh(obs)[0]

In [ ]:
SparsePauliOp.from_operator(obs)

In [ ]:
meas = [np.trace(rdm @ obs) for rdm in rdms]
meas = np.reshape(meas, lattice_shape)

plt.matshow(meas, origin='lower', cmap='plasma', extent=params_extent)
plt.colorbar()

In [ ]:
# Optionally select a rank-1 observable (note the ordering of the eigenvalues).
v0 = np.linalg.eigh(obs)[1][:, 0]
v0 = np.expand_dims(v0, -1)
obs0 = v0 @ v0.T

SparsePauliOp.from_operator(obs0)

In [ ]:
plt.matshow(obs0)
plt.colorbar()

In [ ]:
meas = [np.trace(rdm @ obs0) for rdm in rdms]
meas = np.reshape(meas, lattice_shape)

plt.matshow(meas, origin='lower', cmap='plasma', extent=params_extent)
plt.xlabel('$\\kappa$')
plt.ylabel('h')
plt.colorbar()

In [ ]:
plot_grad_g_angle_stream(grad_g, params_extent=params_extent, theory_lines=False);